<a href="https://colab.research.google.com/github/maorkh4/playground-smashing-ai/blob/main/2_bigrams_exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bigrams Exercises

1. Watch the [bigrams video](https://www.youtube.com/watch?v=PaCmpygFfXo) on YouTube
2. Come back and complete these exercises to level up :)

## Exercise 1: Trigram Model

**Estimated time: 30-45 minutes**

Extend the bigram model to a trigram model (3 characters predict the next one).
You can choose to implement either:
- A counting-based approach (similar to the bigram counting approach)
- A neural network approach (extending the neural bigram model)

Compare the loss and quality of generated names.

In [5]:
# TODO: Implement trigram model
# Your code here

import torch
import torch.nn.functional as F
# start boilerplate
import requests

url = "https://raw.githubusercontent.com/karpathy/makemore/master/names.txt"
response = requests.get(url)
response.raise_for_status() # Raise an exception for bad status codes (4xx or 5xx)

names = response.text.splitlines()
# end boilerplate

stoi = {s:i+1 for i,s in enumerate(sorted(list(set(''.join(names)))))}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}

def build_dataset(words):
  X, Y = [], []

  for w in words:
    context = [0, 0]
    for ch in w + '.':
      ix = stoi[ch]
      X.append(context)
      Y.append(ix)
      context = context[1:] + [ix] # crop and append

  X = torch.tensor(X)
  Y = torch.tensor(Y)
  num = Y.nelement()
  return X, Y, num

Xtr, Ytr, num = build_dataset(names)
print(Xtr.shape, Ytr.shape, num)

g = torch.Generator().manual_seed(2147483647)
W = torch.randn((54, 27), generator=g, requires_grad=True)

torch.Size([228146, 2]) torch.Size([228146]) 228146


In [13]:
xenc = F.one_hot(Xtr, num_classes=27).float().view(Xtr.shape[0], -1)
print(xenc.shape)

for k in range(200):
  # forward pass
  logits = xenc @ W
  counts = logits.exp()
  probs = counts / counts.sum(1, keepdims=True)
  loss = -probs[torch.arange(num), Ytr].log().mean() + 0.01 * (W**2).mean()
  if k % 20 == 0:
    print(f'step {k:3d}: loss = {loss.item():.4f}')

  # backward pass
  W.grad = None
  loss.backward()

  # update
  W.data += -50 * W.grad

print(f'final loss: {loss.item():.4f}')

torch.Size([228146, 54])
3.318845510482788


In [15]:
g = torch.Generator().manual_seed(2147483647)

for i in range(5):

  out = []
  ix = [0, 0]
  while True:
    xenc = F.one_hot(torch.tensor([ix]), num_classes=27).float().view(-1, W.shape[0])
    logits = xenc @ W # predict log-counts
    counts = logits.exp() # counts, equivalent to N
    p = counts / counts.sum(1, keepdims=True) # probabilities for next character
    # ----------
    ix[0] = ix[1]
    ix[1] = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()

    out.append(itos[ix[1]])
    if ix[1] == 0:
      break
  print(''.join(out))

cexze.
mogllupfipqaytydhmkllimjxta.
nrllayn.
kava.
.


## Exercise 2: Train/Dev/Test Split

**Estimated time: 20-30 minutes**

Split the dataset into 80% training, 10% development (validation), and 10% test sets.
- Train your model on the training set
- Tune hyperparameters using the dev set
- Report final performance on the test set

Make sure to shuffle the data before splitting!

In [ ]:
# TODO: Implement 80/10/10 split
# Your code here


## Exercise 3: Tune Smoothing Strength

**Estimated time: 25-35 minutes**

In the counting-based bigram model, we added smoothing to avoid zero probabilities.
Experiment with different smoothing strengths and find the value that gives the best loss on the dev set.

Try values like: 0.01, 0.1, 1, 5, 10, 50, 100, etc.

In [ ]:
# TODO: Tune smoothing parameter
# Your code here


## Exercise 4: Direct Indexing Instead of One-Hot

**Estimated time: 15-25 minutes**

In the neural network bigram model, we used `F.one_hot` to convert integers to one-hot vectors.
This is inefficient! Instead, use direct indexing into the weight matrix.

Hint: If `W` is your weight matrix and `x` is your input integer, you can just do `W[x]` instead of `(F.one_hot(x) @ W)`

In [ ]:
# TODO: Replace F.one_hot with direct indexing
# Your code here


## Exercise 5: Use F.cross_entropy

**Estimated time: 20-30 minutes**

Replace the manual implementation of negative log-likelihood loss with PyTorch's built-in `F.cross_entropy`.

Make sure you understand:
- What inputs `F.cross_entropy` expects
- How it combines softmax and negative log-likelihood
- That you get the same loss value as before

In [ ]:
# TODO: Use F.cross_entropy instead of manual loss
# Your code here


## Exercise 6: Meta-Exercise (Design Your Own)

**Estimated time: 30-60 minutes**

Design and implement your own exercise to explore the bigram model further!

Some ideas:
- Visualize the learned weights/probabilities
- Compare different initialization schemes
- Experiment with learning rate schedules
- Try different architectures (add more neurons, layers, etc.)
- Analyze what character combinations the model finds most likely/unlikely
- Generate names with different temperature settings for sampling

Be creative!

In [ ]:
# TODO: Your own exercise
# Your code here
